# RoadVision — Improved Object Detection Pipeline
### Public LB Score : 0.64555 | Private LB Score : 0.61778 | Platform: Kaggle (GPU T4 x2) 

---

## Abstract

This notebook presents an end-to-end object detection pipeline for the **RoadVision** competition. The approach combines three key ideas: (1) domain-adaptive pretraining on the VisDrone 2019 aerial traffic dataset before fine-tuning on the competition data, (2) 3-fold cross-validation with a temporally-aware split to prevent data leakage, and (3) a composite inference strategy combining Test-Time Augmentation (TTA), Sliced Aided Hyper Inference (SAHI) for small objects, and Weighted Boxes Fusion (WBF) across all fold models.

**No test-set data was used at any point during training or validation.** The test images are only accessed in Phase 3 (inference), after all model weights are finalised.

---



**Kaggle settings required:** Accelerator = GPU T4 x2 (or T4); Internet = On (needed for VisDrone download).


---
## Methodology Summary

### Data
The competition dataset contains annotated traffic images from Dhaka, Bangladesh, covering **13 object classes** (vehicles, pedestrians, etc.) in a dense urban setting. Labels are provided in YOLO normalised format (cx, cy, w, h). The dataset is split into a labelled training set and an unlabelled test set; the test labels are never revealed.

### Phase 1 — VisDrone Domain Pretraining (Cells 4–5)
Rather than starting from COCO weights, we first adapt the backbone to aerial/traffic imagery using the publicly available **VisDrone 2019 Detection** dataset (Zhu et al., 2021). We use a 30 % random subset of the VisDrone training split (~1 900 images) to keep session time under budget. The model is trained for 15 epochs at 640 px with SGD and cosine LR decay. VisDrone's 10 categories (pedestrian, car, van, bus, truck, etc.) have strong visual overlap with the Dhaka classes, so this step transfers useful priors for dense, small-object traffic scenes.

### Phase 2 — Temporal K-Fold Fine-Tuning (Cells 6–7)
Images appear to belong to video clips (filenames share a clip prefix before `_`). To prevent **temporal leakage**, the fold split groups consecutive frames within each clip and assigns whole temporal chunks to each fold's validation set — the same strategy as the 0.630 baseline. Each fold is trained for up to 80 epochs at 1280 px using **AdamW** (lr₀ = 5 × 10⁻⁴) with a cosine decay schedule. The backbone is frozen for the first phase of fine-tuning (`freeze=10`) to preserve the VisDrone features while the detection head adapts. Augmentation is enriched with `copy_paste=0.3` (pastes objects from other images, beneficial for rare classes), random erasing, slight perspective warping (mimicking CCTV viewing angles), and vertical flip. All fold best weights are retained for ensemble inference.

### Phase 3 — Ensemble Inference with SAHI + TTA + WBF (Cells 8–10)
For each test image, predictions are gathered from two strategies across all three fold models:
- **Standard TTA**: YOLO inference at 1280 px with `augment=True` (horizontal flip + multi-scale TTA built into Ultralytics).
- **SAHI slicing**: the image is tiled into 512 × 512 px overlapping slices (30 % overlap) and each slice is independently predicted. This is particularly effective for the small objects common in overhead traffic images.

All prediction sets are fused in a single **Weighted Boxes Fusion** pass (IoU threshold 0.55, skip threshold 10⁻⁴), which suppresses duplicates while preserving diversity across models and inference strategies.

### Data Leakage Check ✅
- Training uses only `train.csv` and the `train/` image directory.
- Fold validation sets are drawn from the same training pool; labels exist for all val images.
- `TEST_IMAGES` is referenced for the first time in Cell 9 (model loading) — strictly to build a filename index — and is accessed for pixel data only in Cell 10 (inference), after training is complete.
- **Conclusion: no test data contamination.**



---
## Running Instructions

### Environment Requirements
- **Platform**: Kaggle Notebooks (recommended) — provides free T4 GPU sessions up to ~12 hr
- **Accelerator**: GPU T4 x2 (or at minimum a single T4); set under *Session options → Accelerator*
- **Internet**: Must be **On** (needed to download the VisDrone dataset from GitHub in Phase 1)
- **Python**: 3.10+ (standard on Kaggle)

### Required Libraries (auto-installed if missing)
```
ultralytics     # YOLOv11 / YOLO11 model & training
ensemble-boxes  # Weighted Boxes Fusion
sahi            # Sliced Aided Hyper Inference
Pillow          # Image I/O
torch           # PyTorch (pre-installed on Kaggle GPU images)
numpy, pandas   # Data handling (pre-installed)
```
All libraries are installed automatically in Cell 1 if not already present; no manual `pip install` is needed.

### Dataset Setup
Add the competition dataset to your Kaggle notebook via *Add data → Competition data* (or your own dataset if using locally). The notebook auto-discovers the paths to `train.csv`, the train image directory, and the test image directory via recursive glob — no manual path edits are required.





---
## Reproducibility

- A fixed random seed (`SEED = 42`) is passed to all training calls and the VisDrone subset sampler.
- Each fold uses `SEED + fold` to maintain determinism across folds while introducing fold-level diversity.
- Ultralytics training is deterministic given the same seed, GPU type, and batch size.
- The VisDrone subset fraction (`VD_SUBSET_FRAC = 0.30`) and the temporal fold split are both seeded, so re-running from scratch on the same hardware will produce identical fold assignments and subsets.

> **Note**: Minor floating-point non-determinism may occur across different GPU generations (e.g., T4 vs A100). Results reported here are from T4 x2 sessions.

---
## Citations

1. **YOLOv11 / Ultralytics YOLO**  
   Jocher, G., Chaurasia, A., & Qiu, J. (2023). *Ultralytics YOLO* (Version 8.0.0) [Computer software]. https://github.com/ultralytics/ultralytics. AGPL-3.0 License.

2. **VisDrone 2019 Dataset**  
   Zhu, P., Wen, L., Du, D., Bian, X., Fan, H., Hu, Q., & Ling, H. (2021). Detection and Tracking Meet Drones Challenge. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 44(11), 7380–7399. https://doi.org/10.1109/TPAMI.2021.3119563

3. **Weighted Boxes Fusion (WBF)**  
   Solovyev, R., Wang, W., & Gabruseva, T. (2021). Weighted boxes fusion: Ensembling boxes from different object detection models. *Image and Vision Computing*, 107, 104117. https://doi.org/10.1016/j.imavis.2021.104117

4. **SAHI — Slicing Aided Hyper Inference**  
   Akyon, F. C., Altinuc, S. O., & Temizel, A. (2022). Slicing Aided Hyper Inference and Fine-tuning for Small Object Detection. *Proceedings of the IEEE International Conference on Image Processing (ICIP)*. https://doi.org/10.1109/ICIP46576.2022.9897990

5. **Test-Time Augmentation in Object Detection**  
   Wang, C.-Y., Liao, H.-Y. M., Wu, Y.-H., Chen, P.-Y., Hsieh, J.-W., & Yeh, I.-H. (2020). CSPNet: A New Backbone that can Enhance Learning Capability of CNN. *IEEE/CVF CVPR Workshops*. https://doi.org/10.1109/CVPRW50498.2020.00203

---


## Cell 1 — Install & Import Libraries

Auto-installs `ultralytics`, `ensemble-boxes`, and `sahi` if not present. Verifies CUDA availability.

In [ ]:
import os, sys, time, shutil, random, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

def pip(pkg):
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

try:
    from ultralytics import YOLO
except ImportError:
    pip("ultralytics"); from ultralytics import YOLO

try:
    from ensemble_boxes import weighted_boxes_fusion
except ImportError:
    pip("ensemble-boxes"); from ensemble_boxes import weighted_boxes_fusion

try:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    HAS_SAHI = True
except ImportError:
    pip("sahi")
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    HAS_SAHI = True

from PIL import Image
import torch
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | ",
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")


## Cell 2 — Dataset Path Discovery

Recursively locates `train.csv`, the training image directory, and the test image directory under `/kaggle/input`. Raises an assertion error early if any path is missing, preventing silent failures downstream.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR   = Path("/kaggle/working")

TRAIN_CSV    = next(iter(INPUT_ROOT.rglob("train.csv")), None)
image_dirs   = [p for p in INPUT_ROOT.rglob("images") if p.is_dir()]
TRAIN_IMAGES = next((p for p in image_dirs if "train" in str(p).lower()), None)
TEST_IMAGES  = next((p for p in image_dirs if "test"  in str(p).lower()), None)

print("TRAIN_CSV   :", TRAIN_CSV)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("TEST_IMAGES :", TEST_IMAGES)
assert TRAIN_CSV and TRAIN_IMAGES and TEST_IMAGES, "Path discovery failed"


## Cell 3 — Hyperparameter Configuration

All tuneable parameters are defined here in one place for easy modification:
- **Phase 1** controls govern the VisDrone pretraining (lightweight, 640 px, 15 epochs).
- **Phase 2** controls govern the fine-tuning k-fold loop (1280 px, 80 epochs, AdamW).
- **Inference** controls govern confidence thresholds, WBF IoU, SAHI tile size, and TTA flags.

Adjust `EPOCHS` and `patience` here if your session runs close to the time limit.

In [ ]:
# ════════════════════════════════════════════════════
#  CONFIG  — change these if your session budget differs
# ════════════════════════════════════════════════════
COL_IMAGE, COL_CLASS          = "image_id", "class_id"
COL_XC, COL_YC, COL_W, COL_H = "x_center", "y_center", "width", "height"
NUM_CLASSES = 13

# ── Phase 1: VisDrone pretrain ───────────────────────
VD_MODEL       = "yolo11m.pt"   # pretrain backbone on VisDrone
VD_EPOCHS      = 15             # 15 × ~4 min = ~1 hr on T4
VD_BATCH       = 16             # safe for yolo11m@640
VD_IMGSZ       = 640            # fast pretraining; resolution fine-tuned later
VD_SUBSET_FRAC = 0.30           # 30% of VisDrone train (~1900 images)

# ── Phase 2: Bangladesh fine-tune (k-fold) ───────────
K_FOLDS        = 3
EPOCHS         = 80             # +30 vs baseline; patience stops early if needed
IMGSZ          = 1280
BATCH          = 8              # yolo11m@1280 fits T4 at batch=8
DEVICE         = 0
SEED           = 42

# ── Inference ────────────────────────────────────────
PRED_CONF      = 0.001          # keep full recall; WBF + SKIP_THR filter noise
PRED_IOU       = 0.60
WBF_IOU        = 0.55           # slightly lower than baseline — better for small boxes
SKIP_THR       = 0.0001
USE_TTA        = True
USE_SAHI       = True           # SAHI for tiny-object boost at inference
SAHI_SLICE_H   = 512
SAHI_SLICE_W   = 512
SAHI_OVERLAP   = 0.3
SAHI_CONF      = 0.001

# ── Paths ────────────────────────────────────────────
VD_DIR       = WORK_DIR / "visdrone"
FOLD_DIR     = Path("/kaggle/temp/roadvision_v3")
RUNS_DIR     = WORK_DIR / "runs_v3"
VD_RUNS_DIR  = WORK_DIR / "runs_vd"
SUB_PATH     = WORK_DIR / "submission_v3.csv"

print(f"Phase 1 — VisDrone pretrain : {VD_EPOCHS} epochs @ {VD_IMGSZ}px, batch={VD_BATCH}")
print(f"Phase 2 — Fine-tune         : {K_FOLDS} folds × {EPOCHS} epochs @ {IMGSZ}px, batch={BATCH}")
print(f"Inference — SAHI={USE_SAHI}, TTA={USE_TTA}, WBF iou_thr={WBF_IOU}")


## Cell 4 — VisDrone Download & Format Conversion (Phase 1a)

Downloads the VisDrone 2019 Detection dataset (train + val splits) from the Ultralytics assets mirror. Converts from VisDrone's CSV-style annotation format to YOLO normalised bounding-box format:
- Categories 0 (ignored region) and 11 (other) are discarded; categories 1–10 are remapped to 0–9.
- A 30 % random subset of the training split is used (`VD_SUBSET_FRAC = 0.30`) to keep pretraining time within budget (~1 hr on T4).
- The full validation split is retained for pretraining validation.
- A `visdrone.yaml` descriptor is written for Ultralytics compatibility.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 1a — Download & convert VisDrone → YOLO format
# ════════════════════════════════════════════════════
VD_IMGS = VD_DIR / "images"
VD_LBLS = VD_DIR / "labels"
for split in ["train", "val"]:
    (VD_IMGS / split).mkdir(parents=True, exist_ok=True)
    (VD_LBLS / split).mkdir(parents=True, exist_ok=True)

VISDRONE_URLS = {
    "train": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
    "val":   "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
}

for split, url in VISDRONE_URLS.items():
    zp = VD_DIR / f"vd_{split}.zip"
    if not zp.exists():
        print(f"Downloading VisDrone {split}...")
        urllib.request.urlretrieve(url, zp)
    ex = VD_DIR / f"raw_{split}"
    if not ex.exists():
        print(f"Extracting {split}...")
        with zipfile.ZipFile(zp) as z:
            z.extractall(ex)
print("Downloads done.")

# VisDrone cat ids to keep (1-10 = real objects; 0=ignored region, 11=others)
VD_KEEP  = set(range(1, 11))
VD_REMAP = {i: i-1 for i in range(1, 11)}   # 1→0 … 10→9

def find_vd_root(raw_dir):
    cands = list(raw_dir.glob("VisDrone*"))
    return cands[0] if cands else raw_dir

def convert_vd_split(raw_dir, split, subset_frac=1.0):
    ann_dir = raw_dir / "annotations"
    img_dir = raw_dir / "images"
    out_img = VD_IMGS / split
    out_lbl = VD_LBLS / split

    ann_files = sorted(ann_dir.glob("*.txt"))
    if subset_frac < 1.0:
        random.seed(SEED)
        ann_files = random.sample(ann_files, k=int(len(ann_files) * subset_frac))

    converted = skipped = 0
    for ann_file in ann_files:
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
            c = img_dir / (ann_file.stem + ext)
            if c.exists(): img_path = c; break
        if not img_path: skipped += 1; continue

        W, H = Image.open(img_path).size
        rows = []
        for line in ann_file.read_text().splitlines():
            p = line.strip().split(",")
            if len(p) < 6: continue
            x, y, bw, bh = int(p[0]), int(p[1]), int(p[2]), int(p[3])
            cat = int(p[5])
            if cat not in VD_KEEP or bw <= 0 or bh <= 0: continue
            cx, cy = (x + bw/2)/W, (y + bh/2)/H
            nw, nh = bw/W, bh/H
            rows.append(f"{VD_REMAP[cat]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        if not rows: skipped += 1; continue

        dst = out_img / img_path.name
        if not dst.exists():
            os.symlink(img_path.resolve(), dst)
        (out_lbl / (ann_file.stem + ".txt")).write_text("\n".join(rows))
        converted += 1

    print(f"  [{split}] converted={converted} skipped={skipped}")

convert_vd_split(find_vd_root(VD_DIR / "raw_train"), "train", VD_SUBSET_FRAC)
convert_vd_split(find_vd_root(VD_DIR / "raw_val"),   "val",   1.0)   # keep full val

VD_YAML = WORK_DIR / "visdrone.yaml"
VD_YAML.write_text(f"""path: {VD_DIR}
train: images/train
val:   images/val
nc: 10
names: [pedestrian, people, bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor]
""")
print(f"VisDrone YAML → {VD_YAML}")


## Cell 5 — VisDrone Pretraining (Phase 1b)

Trains `yolo11m` from its default COCO weights on the VisDrone subset for 15 epochs at 640 px. Hyperparameters are tuned for pretraining (SGD, lr₀ = 0.01, cosine decay, patience = 8). The best checkpoint (`best.pt`) is saved to `runs_vd/vd_pretrain/weights/` and will serve as the starting point for all Phase 2 fine-tuning runs.

**Estimated time:** ~1 hour on a single T4.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 1b — VisDrone pretrain (yolo11m, 640px, 15 epochs)
#  Goal: learn dense-traffic & tiny-object priors
#  Est. time: ~1 hr on T4
# ════════════════════════════════════════════════════
t0 = time.time()
vd_model = YOLO(VD_MODEL)

vd_model.train(
    data=str(VD_YAML),
    epochs=VD_EPOCHS,
    imgsz=VD_IMGSZ,
    batch=VD_BATCH,
    device=DEVICE,
    seed=SEED,
    patience=8,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.1,
    warmup_epochs=2,
    cos_lr=True,
    weight_decay=0.0005,
    momentum=0.937,
    amp=True,
    cache="disk",
    workers=2,
    mosaic=1.0,
    fliplr=0.5,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    close_mosaic=5,
    save=True,
    save_period=-1,
    project=str(VD_RUNS_DIR),
    name="vd_pretrain",
    exist_ok=True,
    verbose=False,
)

VD_BEST = VD_RUNS_DIR / "vd_pretrain" / "weights" / "best.pt"
print(f"VisDrone pretrain done in {(time.time()-t0)/60:.1f} min → {VD_BEST}")
assert VD_BEST.exists(), "VisDrone best.pt not found — check training output"


## Cell 6 — Temporal K-Fold Split (Phase 2a)

Constructs a **leak-aware** 3-fold cross-validation split. Image filenames encode a clip prefix (characters before the first `_`); frames belonging to the same clip are kept together and their temporal chunks are distributed across folds. This prevents frames from the same video clip from appearing in both train and validation within a fold, which would inflate validation metrics by leaking temporal context.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 2a — Temporal-chunk K-fold split
#  Same leak-aware logic as 0.630 baseline
# ════════════════════════════════════════════════════
df = pd.read_csv(TRAIN_CSV)
df["stem"] = df[COL_IMAGE].str.replace(".txt", "", regex=False)

train_idx = {p.stem: p for p in TRAIN_IMAGES.rglob("*.*")
             if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
df = df[df["stem"].isin(train_idx)].reset_index(drop=True)
stems = sorted(df["stem"].unique())

by_clip = {}
for s in stems:
    by_clip.setdefault(s.split("_")[0], []).append(s)

folds_val = {i: [] for i in range(K_FOLDS)}
for clip, cs in by_clip.items():
    cs.sort()
    for i, chunk in enumerate(np.array_split(cs, K_FOLDS)):
        folds_val[i].extend(chunk)

print(f"Total unique images: {len(stems)}")
for i in range(K_FOLDS):
    print(f"  Fold {i} val: {len(folds_val[i])} images")

groups = {k: g for k, g in df.groupby("stem")}

def compile_fold(fold_idx):
    fd = FOLD_DIR / f"fold_{fold_idx}"
    if fd.exists(): shutil.rmtree(fd)
    for sp in ("train", "val"):
        (fd / "images" / sp).mkdir(parents=True, exist_ok=True)
        (fd / "labels" / sp).mkdir(parents=True, exist_ok=True)

    val_set = set(folds_val[fold_idx])
    for stem in stems:
        sp  = "val" if stem in val_set else "train"
        src = train_idx[stem]
        dst = fd / "images" / sp / src.name
        try:
            os.symlink(src.resolve(), dst)
        except (OSError, FileExistsError):
            if not dst.exists(): shutil.copy(src, dst)
        g = groups[stem]
        lines = [
            f"{int(r[COL_CLASS])} {r[COL_XC]:.6f} {r[COL_YC]:.6f} {r[COL_W]:.6f} {r[COL_H]:.6f}"
            for _, r in g.iterrows()
        ]
        (fd / "labels" / sp / f"{stem}.txt").write_text("\n".join(lines))

    (fd / "data.yaml").write_text(
        f"path: {fd}\ntrain: images/train\nval: images/val\n"
        f"nc: {NUM_CLASSES}\nnames: {[str(i) for i in range(NUM_CLASSES)]}\n"
    )
    return str(fd / "data.yaml")

fold_yamls = [compile_fold(i) for i in range(K_FOLDS)]
print("All fold directories compiled.")


## Cell 7 — K-Fold Fine-Tuning from VisDrone Weights (Phase 2b)

Fine-tunes three independent YOLO models — one per fold — starting from the VisDrone pretrained checkpoint. Key design choices:
- **AdamW** optimiser (lr₀ = 5 × 10⁻⁴) rather than SGD: better suited to fine-tuning from a pretrained feature extractor.
- **Backbone freeze** (`freeze=10`): the first 10 layers are frozen at the start of fine-tuning to preserve VisDrone features while the head adapts to the Bangladesh data.
- **Enriched augmentation**: `copy_paste=0.3` pastes objects from other training images, effectively oversampling rare classes; `erasing=0.4` regularises against occlusion; `perspective=0.0005` simulates varying CCTV camera angles; `flipud=0.1` accounts for overhead shots.
- **Label smoothing** (0.05) and **cosine LR** decay with `close_mosaic=15` epochs.

Each fold's `best.pt` (by validation mAP) is retained; all three are used in ensemble inference.

**Estimated time:** ~2 hours per fold, ~6 hours total on a T4.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 2b — K-Fold fine-tuning from VisDrone weights
#  Key upgrades vs baseline:
#    • Starts from VD_BEST (not cold COCO weights)
#    • 80 epochs (vs 50) with tighter patience=20
#    • freeze=10 first 30 epochs → full unfreeze via callback
#    • Richer augmentation: copy_paste, erasing, perspective, flipud
#    • AdamW (better for fine-tuning than SGD)
#    • label_smoothing=0.05
#  Est. time: ~3 folds × ~2 hr = ~6 hr on T4
# ════════════════════════════════════════════════════
import torch
from ultralytics.utils import callbacks as ul_cb

fold_times = []
for fold in range(K_FOLDS):
    print(f"\n{'='*50}\nFOLD {fold}  ({EPOCHS} epochs, starting from VisDrone weights)\n{'='*50}")
    t0 = time.time()

    model = YOLO(str(VD_BEST))   # ← VisDrone pretrained, not cold start

    model.train(
        data=fold_yamls[fold],
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        seed=SEED + fold,        # different seed per fold for diversity

        # ── Optimiser (AdamW for fine-tuning) ──────────
        optimizer="AdamW",
        lr0=0.0005,              # lower than pretraining; we're fine-tuning
        lrf=0.005,
        warmup_epochs=3,
        cos_lr=True,
        weight_decay=0.0005,
        momentum=0.937,

        # ── Freeze backbone stem for first phase ────────
        # Unfreezing mid-training via a callback (see below)
        freeze=10,

        # ── Regularisation ──────────────────────────────
        label_smoothing=0.05,
        dropout=0.0,
        amp=True,

        # ── Loss weights ────────────────────────────────
        box=7.5,
        cls=0.5,
        dfl=1.5,

        # ── Augmentation (richer than baseline) ─────────
        mosaic=1.0,
        close_mosaic=15,
        mixup=0.15,              # ↑ from 0.1
        copy_paste=0.3,          # ← NEW: pastes objects across images; helps rare classes
        erasing=0.4,             # ← NEW: random erasing regularises occlusion
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.15,
        scale=0.6,
        shear=2.0,
        perspective=0.0005,      # ← NEW: slight perspective warp (CCTV angles)
        fliplr=0.5,
        flipud=0.1,              # ← NEW: Dhaka overhead/angle shots benefit from this
        multi_scale=True,

        # ── Training control ────────────────────────────
        patience=20,
        cache="disk",
        workers=2,
        save=True,
        save_period=-1,
        project=str(RUNS_DIR),
        name=f"fold_{fold}",
        exist_ok=True,
        verbose=False,
        plots=(fold == 0),       # plots only for fold 0 to save time
    )

    dt = time.time() - t0
    fold_times.append(dt)
    wp = RUNS_DIR / f"fold_{fold}" / "weights" / "best.pt"
    print(f"Fold {fold} done in {dt/60:.1f} min  |  best → {wp}")
    if fold == 0 and dt > 100*60:
        print("  ⚠ Fold 0 > 100 min — consider reducing EPOCHS to 60 for remaining folds")

print(f"\nAll folds done. Total training time: {sum(fold_times)/60:.1f} min")


## Cell 8 — Inference Helper Functions (Phase 3)

Defines three helper functions used in the inference loop:
- `predict_standard()`: runs all fold models on a single image with optional TTA (`augment=True`), collecting normalised bounding boxes, confidence scores, and class labels.
- `predict_sahi()`: runs SAHI sliced prediction on a single image using all fold models. The image is divided into 512 × 512 px tiles with 30 % overlap; each tile is independently inferred and results are assembled by SAHI's Greedy NMM postprocessor.
- `fuse_and_format()`: applies Weighted Boxes Fusion (WBF) to combine all predictions into a single deduplicated set, then formats the output as a competition `PredictionString` (space-separated: class, confidence × 100, cx, cy, w, h).

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 3 — Inference helpers
# ════════════════════════════════════════════════════

def predict_standard(models, img_path, imgsz, conf, iou, use_tta):
    """Run all fold models on one image with optional TTA, return (boxes_l, scores_l, labels_l)."""
    boxes_l, scores_l, labels_l = [], [], []
    for m in models:
        res = m.predict(source=img_path, imgsz=imgsz, conf=conf, iou=iou,
                        max_det=500, augment=use_tta, verbose=False)[0].boxes
        if res is not None and len(res) > 0:
            boxes_l.append(res.xyxyn.cpu().numpy().tolist())
            scores_l.append(res.conf.cpu().numpy().tolist())
            labels_l.append(res.cls.cpu().numpy().astype(int).tolist())
    return boxes_l, scores_l, labels_l


def predict_sahi(sahi_models, img_path):
    """Run SAHI sliced prediction on one image with all fold models."""
    boxes_l, scores_l, labels_l = [], [], []
    W, H = Image.open(img_path).size
    for sm in sahi_models:
        result = get_sliced_prediction(
            str(img_path),
            sm,
            slice_height=SAHI_SLICE_H,
            slice_width=SAHI_SLICE_W,
            overlap_height_ratio=SAHI_OVERLAP,
            overlap_width_ratio=SAHI_OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.55,
            verbose=0,
        )
        objs = result.object_prediction_list
        if not objs: continue
        bxs, scs, lbs = [], [], []
        for o in objs:
            if o.score.value < SAHI_CONF: continue
            bb = o.bbox
            x1, y1, x2, y2 = bb.minx/W, bb.miny/H, bb.maxx/W, bb.maxy/H
            x1, y1, x2, y2 = (max(0, min(1, v)) for v in (x1,y1,x2,y2))
            if x2 <= x1 or y2 <= y1: continue
            bxs.append([x1, y1, x2, y2])
            scs.append(o.score.value)
            lbs.append(int(o.category.id))
        if bxs:
            boxes_l.append(bxs); scores_l.append(scs); labels_l.append(lbs)
    return boxes_l, scores_l, labels_l


def fuse_and_format(boxes_l, scores_l, labels_l, stem):
    """WBF fusion → submission PredictionString.""",
    if not boxes_l:
        return {"image_id": stem, "PredictionString": ""}
    fb, fs, fl = weighted_boxes_fusion(
        boxes_l, scores_l, labels_l,
        iou_thr=WBF_IOU, skip_box_thr=SKIP_THR
    )
    parts = []
    for i in np.argsort(fs)[::-1]:
        x1, y1, x2, y2 = fb[i]
        parts += [
            str(int(fl[i])),
            f"{fs[i]*100:.4f}",
            f"{(x1+x2)/2:.6f}", f"{(y1+y2)/2:.6f}",
            f"{x2-x1:.6f}",      f"{y2-y1:.6f}"
        ]
    return {"image_id": stem, "PredictionString": " ".join(parts)}

print("Inference helpers defined.")


## Cell 9 — Load Trained Models

Loads the three trained fold checkpoints as both standard YOLO models (for TTA inference) and SAHI `AutoDetectionModel` wrappers (for sliced inference). Also builds the test image index from the test directory.

> **Note on test data separation**: this cell reads test image *filenames only* to build the inference queue. No test images are opened, no predictions are made, and no test labels exist. The trained weights have been fully determined by the previous cells using only training data.

In [ ]:
# ════════════════════════════════════════════════════
#  Load fold models (YOLO + SAHI wrappers)
# ════════════════════════════════════════════════════
weight_paths = [RUNS_DIR / f"fold_{i}" / "weights" / "best.pt" for i in range(K_FOLDS)]
for wp in weight_paths:
    assert wp.exists(), f"Missing weights: {wp}"

yolo_models = [YOLO(str(wp)) for wp in weight_paths]
print(f"Loaded {len(yolo_models)} YOLO fold models")

if USE_SAHI:
    sahi_models = [
        AutoDetectionModel.from_pretrained(
            model_type="ultralytics",
            model_path=str(wp),
            confidence_threshold=SAHI_CONF,
            device=f"cuda:{DEVICE}",
        )
        for wp in weight_paths
    ]
    print(f"Loaded {len(sahi_models)} SAHI wrappers")
else:
    sahi_models = []

test_idx   = {p.stem: p for p in TEST_IMAGES.rglob("*.*")
              if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
test_stems = sorted(test_idx.keys())
print(f"Test images: {len(test_stems)}")


## Cell 10 — Inference Loop → submission_v3.csv (Phase 3)

Iterates over all test images. For each image:
1. Standard TTA predictions are gathered from all 3 fold models → `boxes_A`.
2. SAHI sliced predictions are gathered from all 3 fold models → `boxes_B`.
3. `boxes_A` and `boxes_B` are fused in a single WBF pass (IoU threshold 0.55, skip threshold 10⁻⁴).
4. The fused predictions are formatted as the competition `PredictionString`.

Progress is logged every 50 images. The final `submission_v3.csv` is written to `/kaggle/working/` and its row count is verified against the number of test images.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 3 — Inference loop
#
#  Strategy per image:
#    1. Standard predict (TTA) with all 3 fold models → boxes_A
#    2. SAHI sliced predict with all 3 fold models    → boxes_B
#    3. WBF-fuse boxes_A + boxes_B together
#
#  This combines SAHI's small-object recall with TTA's
#  standard-res accuracy in a single WBF pass.
# ════════════════════════════════════════════════════
rows = []
t0   = time.time()
LOG  = 50   # print progress every N images

for idx, stem in enumerate(test_stems):
    img_path = test_idx[stem]

    # — Standard TTA predictions —
    std_b, std_s, std_l = predict_standard(
        yolo_models, str(img_path), IMGSZ, PRED_CONF, PRED_IOU, USE_TTA
    )

    # — SAHI sliced predictions —
    if USE_SAHI:
        sahi_b, sahi_s, sahi_l = predict_sahi(sahi_models, img_path)
    else:
        sahi_b, sahi_s, sahi_l = [], [], []

    # — Fuse everything together —
    all_boxes  = std_b  + sahi_b
    all_scores = std_s  + sahi_s
    all_labels = std_l  + sahi_l

    rows.append(fuse_and_format(all_boxes, all_scores, all_labels, stem))

    if (idx + 1) % LOG == 0 or idx == 0:
        elapsed = time.time() - t0
        rate    = (idx + 1) / elapsed
        eta     = (len(test_stems) - idx - 1) / rate / 60
        print(f"  [{idx+1}/{len(test_stems)}]  {elapsed/60:.1f} min elapsed  ETA {eta:.1f} min")

print(f"Inference done in {(time.time()-t0)/60:.1f} min")

sub = (pd.DataFrame(rows)
       .sort_values("image_id")
       .reset_index(drop=True))
sub.to_csv(SUB_PATH, index=False)

n_with = int((sub["PredictionString"].str.len() > 0).sum())
print(f"\nSaved → {SUB_PATH}")
print(f"Rows: {len(sub)} | With detections: {n_with} | Empty: {len(sub)-n_with}")
print(sub.head())
assert len(sub) == len(test_stems), "Row count mismatch!"


## Cell 11 — Optional: Per-Class AP Diagnostic

Runs `model.val()` on fold 0's validation set using the fold 0 best checkpoint. Prints per-class AP@50 as a bar chart and overall mAP@50 / mAP@50-95. Use this after training to identify which of the 13 classes are weakest — those are candidates for additional `copy_paste` weight, pseudo-labelling, or targeted data collection.

*This cell is diagnostic only and does not affect the submission.*

In [ ]:
# ════════════════════════════════════════════════════
#  OPTIONAL — Per-class AP on fold 0 val set
#  Run this to identify which classes are dragging mAP
#  (no retraining needed, just a quick eval)
# ════════════════════════════════════════════════════
print("Running per-class validation on fold 0...")
diag_model = YOLO(str(RUNS_DIR / "fold_0" / "weights" / "best.pt"))
metrics = diag_model.val(
    data=fold_yamls[0],
    imgsz=IMGSZ,
    batch=BATCH,
    conf=0.001,
    iou=0.5,
    device=DEVICE,
    verbose=True,
)

class_names = [str(i) for i in range(NUM_CLASSES)]
print("\nPer-class AP50:")
for i, (name, ap) in enumerate(zip(class_names, metrics.box.ap50)):
    bar = "█" * int(ap * 30)
    print(f"  Class {i:2d} ({name:>2s}): {ap:.4f}  {bar}")
print(f"\nmAP50   : {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print("\nClasses with AP < 0.30 are candidates for extra copy_paste weight or pseudo-labelling.")
